## Selected Research Question:
"Do genes responsible for first-pass drug metabolism (such as CYP enzymes) show
significantly lower expression in cirrhotic livers compared to healthy livers?"
## Project Objectives:
1. To acquire and preprocess a public gene expression dataset comparing healthy and
cirrhotic human livers from NCBI GEO.
2. To explore and visualize the data by comparing the expression levels of key
drug-metabolizing genes (such as CYP3A4) across healthy and diseased samples.
3. To train and evaluate a machine learning classification model to predict liver tissue
states based on these metabolic gene profiles.
4. To interpret the results and conclude how liver cirrhosis impacts drug-metabolizing
gene expression and first-pass metabolic capacity.

In [ ]:
! pip install GEOparse pydeseq2 pandas numpy matplotlib seaborn

## DATA ACQUSITION

In [2]:
import GEOparse
import pandas as pd
import numpy as np
from pydeseq2.ds import DeseqDataSet, DeseqStats

def inspect_geo_series(accession_id):
    print(f"\n--- Inspecting {accession_id} ---")
    gse = GEOparse.get_GEO(geo=accession_id, destdir="./geodata", annotate_gpl=True)
    platforms = list(gse.gpl.keys()) if hasattr(gse, 'gpl') else "Unknown"
    print(f"Platform(s) (GPL): {platforms}")
    print(f"Total samples:{len(gse.gsms)}")

    meta_list = []
    for gsm_name, gsm in  gse.gsms.items():
        source = gsm.metadata.get("source_name_ch1",[ "Unknown"])[0]
        title = gsm.metadata.get("title",['Unknown'])[0]
        meta_list.append({'sample':gsm_name, 'title':title, 'source':source})
    meta_df = pd.DataFrame(meta_list)
    meta_df['predicted_group'] = meta_df['source'].apply(
        lambda x:'cirrhosis' if any(term in x.lower() for term in ['cirrhosis', 'cirrhotic', 'fibrosis'])
        else ('healthy' if any(term in x.lower() for term in ['normal', 'healthy', 'control']) else 'unclassified')

    )
    print ("Group Breakdown Preview:")
    print (meta_df['predicted_group'].value_counts())
    return gse, meta_df



/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [3]:
gse_candidate, meta_candidate = inspect_geo_series("GSE25097")
unclassified_df = meta_candidate[meta_candidate['predicted_group'] == 'unclassified']
print(unclassified_df['source'].value_counts())

05-Nov-2026 09:40:11 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/GSE25097/soft/GSE25097_family.soft.gz to ./geodata/GSE25097_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/GSE25097/soft/GSE25097_family.soft.gz to ./geodata/GSE25097_family.soft.gz



--- Inspecting GSE25097 ---


100%|██████████| 168M/168M [00:01<00:00, 155MB/s]
05-Nov-2026 09:40:13 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
05-Nov-2026 09:40:13 DEBUG downloader - Moving /tmp/tmpv224r3my to /content/geodata/GSE25097_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpv224r3my to /content/geodata/GSE25097_family.soft.gz
05-Nov-2026 09:40:13 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/GSE25097/soft/GSE25097_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/GSE25097/soft/GSE25097_family.soft.gz
05-Nov-2026 09:40:13 INFO GEOparse - Parsing ./geodata/GSE25097_family.soft.gz: 
INFO:GEOparse:Parsing ./geodata/GSE25097_family.soft.gz: 
05-Nov-2026 09:40:13 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
05-Nov-2026 09:40:13 DEBUG GEOparse - SERIES: GSE25097
DEBUG:GEOparse:SERIES: GSE25097
05-Nov-2026 09:40:13 DEBUG GEOparse - PLATFORM: GPL10687
DEBUG:GEOpa

Platform(s) (GPL): Unknown
Total samples:557
Group Breakdown Preview:
predicted_group
unclassified    511
cirrhosis        40
healthy           6
Name: count, dtype: int64
source
tumor        268
non_tumor    243
Name: count, dtype: int64


FROM GSE25097: Total samples:557
Group Breakdown Preview:
predicted_group
unclassified    511
cirrhosis        40
healthy           6
Name: count, dtype: int64
source
tumor        268
non_tumor    243
Name: count, dtype: int64

In [4]:
gse_candidate, meta_candidate = inspect_geo_series("GSE14323")
unclassified_df = meta_candidate[meta_candidate['predicted_group'] == 'unclassified']
print(unclassified_df['source'].value_counts())

05-Nov-2026 09:40:58 DEBUG utils - Directory ./geodata already exists. Skipping.
DEBUG:GEOparse:Directory ./geodata already exists. Skipping.
05-Nov-2026 09:40:58 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14323/soft/GSE14323_family.soft.gz to ./geodata/GSE14323_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14323/soft/GSE14323_family.soft.gz to ./geodata/GSE14323_family.soft.gz



--- Inspecting GSE14323 ---


100%|██████████| 34.6M/34.6M [00:00<00:00, 66.9MB/s]
05-Nov-2026 09:40:58 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
05-Nov-2026 09:40:58 DEBUG downloader - Moving /tmp/tmpln46m1nx to /content/geodata/GSE14323_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpln46m1nx to /content/geodata/GSE14323_family.soft.gz
05-Nov-2026 09:40:58 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14323/soft/GSE14323_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14323/soft/GSE14323_family.soft.gz
05-Nov-2026 09:40:58 INFO GEOparse - Parsing ./geodata/GSE14323_family.soft.gz: 
INFO:GEOparse:Parsing ./geodata/GSE14323_family.soft.gz: 
05-Nov-2026 09:40:58 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
05-Nov-2026 09:40:58 DEBUG GEOparse - SERIES: GSE14323
DEBUG:GEOparse:SERIES: GSE14323
05-Nov-2026 09:40:58 DEBUG GEOparse - PLATFORM: GPL96
DEBUG:GEOpa

Platform(s) (GPL): Unknown
Total samples:124
Group Breakdown Preview:
predicted_group
cirrhosis       58
unclassified    47
healthy         19
Name: count, dtype: int64
source
liver tissue with HCC    47
Name: count, dtype: int64


FROM GSE14323 Platform(s) (GPL): Unknown
Total samples:124
Group Breakdown Preview:
predicted_group
cirrhosis       58
unclassified    47
healthy         19
Name: count, dtype: int64
source
liver tissue with HCC    47
Name: count, dtype: int64

In [5]:
gse_candidate, meta_candidate = inspect_geo_series("GSE6764")
unclassified_df = meta_candidate[meta_candidate['predicted_group'] == 'unclassified']
print(unclassified_df['source'].value_counts())

05-Nov-2026 09:41:05 DEBUG utils - Directory ./geodata already exists. Skipping.
DEBUG:GEOparse:Directory ./geodata already exists. Skipping.
05-Nov-2026 09:41:05 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6764/soft/GSE6764_family.soft.gz to ./geodata/GSE6764_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6764/soft/GSE6764_family.soft.gz to ./geodata/GSE6764_family.soft.gz



--- Inspecting GSE6764 ---


100%|██████████| 36.9M/36.9M [00:00<00:00, 73.2MB/s]
05-Nov-2026 09:41:05 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
05-Nov-2026 09:41:05 DEBUG downloader - Moving /tmp/tmpkqzn8il5 to /content/geodata/GSE6764_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpkqzn8il5 to /content/geodata/GSE6764_family.soft.gz
05-Nov-2026 09:41:05 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6764/soft/GSE6764_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6764/soft/GSE6764_family.soft.gz
05-Nov-2026 09:41:05 INFO GEOparse - Parsing ./geodata/GSE6764_family.soft.gz: 
INFO:GEOparse:Parsing ./geodata/GSE6764_family.soft.gz: 
05-Nov-2026 09:41:05 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
05-Nov-2026 09:41:05 DEBUG GEOparse - SERIES: GSE6764
DEBUG:GEOparse:SERIES: GSE6764
05-Nov-2026 09:41:05 DEBUG GEOparse - PLATFORM: GPL570
DEBUG:GEOparse:PLATFOR

Platform(s) (GPL): Unknown
Total samples:75
Group Breakdown Preview:
predicted_group
unclassified    52
cirrhosis       13
healthy         10
Name: count, dtype: int64
source
human liver; low-grade dysplastic liver tissue     10
human liver; very advanced HCC                     10
human liver; early HCC                             10
human liver; very early HCC                         8
human liver; high-grade dysplastic liver tissue     7
human liver; advanced HCC                           7
Name: count, dtype: int64


FROM GSE6764 Platform(s) (GPL): Unknown
Total samples:75
Group Breakdown Preview:
predicted_group
unclassified    52
cirrhosis       13
healthy         10
Name: count, dtype: int64
source
human liver; low-grade dysplastic liver tissue     10
human liver; very advanced HCC                     10
human liver; early HCC                             10
human liver; very early HCC                         8
human liver; high-grade dysplastic liver tissue     7
human liver; advanced HCC                           7
Name: count, dtype: int64

In [6]:
gse_139602, meta_139602 = inspect_geo_series("GSE139602")
unclassified_df = meta_139602[meta_139602['predicted_group'] == 'unclassified']
print(unclassified_df['source'].value_counts())


05-Nov-2026 09:41:14 DEBUG utils - Directory ./geodata already exists. Skipping.
DEBUG:GEOparse:Directory ./geodata already exists. Skipping.
05-Nov-2026 09:41:14 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE139nnn/GSE139602/soft/GSE139602_family.soft.gz to ./geodata/GSE139602_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE139nnn/GSE139602/soft/GSE139602_family.soft.gz to ./geodata/GSE139602_family.soft.gz



--- Inspecting GSE139602 ---


100%|██████████| 37.7M/37.7M [00:00<00:00, 75.6MB/s]
05-Nov-2026 09:41:14 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
05-Nov-2026 09:41:14 DEBUG downloader - Moving /tmp/tmp1pliy2y_ to /content/geodata/GSE139602_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmp1pliy2y_ to /content/geodata/GSE139602_family.soft.gz
05-Nov-2026 09:41:14 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE139nnn/GSE139602/soft/GSE139602_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE139nnn/GSE139602/soft/GSE139602_family.soft.gz
05-Nov-2026 09:41:14 INFO GEOparse - Parsing ./geodata/GSE139602_family.soft.gz: 
INFO:GEOparse:Parsing ./geodata/GSE139602_family.soft.gz: 
05-Nov-2026 09:41:14 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
05-Nov-2026 09:41:14 DEBUG GEOparse - SERIES: GSE139602
DEBUG:GEOparse:SERIES: GSE139602
05-Nov-2026 09:41:14 DEBUG GEOparse - PLATFORM: GPL13

Platform(s) (GPL): Unknown
Total samples:39
Group Breakdown Preview:
predicted_group
unclassified    39
Name: count, dtype: int64
source
Liver Biopsy    39
Name: count, dtype: int64


FROM GSE139602
Total samples:39
Group Breakdown Preview:
predicted_group
unclassified    39
Name: count, dtype: int64
source
Liver Biopsy    39
Name: count, dtype: int64






In [7]:
# Inspect all available metadata keys and sample titles for GSE139602
meta_details = []
for gsm_name, gsm in gse_139602.gsms.items():
    title = gsm.metadata.get('title', [''])[0]
    characteristics = gsm.metadata.get('characteristics_ch1', [])
    meta_details.append({'sample': gsm_name, 'title': title, 'char': str(characteristics)})

details_df = pd.DataFrame(meta_details)
print(details_df.head(10))

       sample                        title                        char
0  GSM4144550  Healthy Liver biopsy (LMS1)  ['disease state: Healthy']
1  GSM4144551  Healthy Liver biopsy (LMS2)  ['disease state: Healthy']
2  GSM4144552  Healthy Liver biopsy (LMS3)  ['disease state: Healthy']
3  GSM4144553  Healthy Liver biopsy (LMS4)  ['disease state: Healthy']
4  GSM4144554  Healthy Liver biopsy (LMS5)  ['disease state: Healthy']
5  GSM4144555  Healthy Liver biopsy (LMS6)  ['disease state: Healthy']
6  GSM4144556    eCLD Liver biopsy (LM039)     ['disease state: eCLD']
7  GSM4144557    eCLD Liver biopsy (LM040)     ['disease state: eCLD']
8  GSM4144558    eCLD Liver biopsy (LM118)     ['disease state: eCLD']
9  GSM4144559    eCLD Liver biopsy (LM071)     ['disease state: eCLD']


### GEO Candidate Dataset Evaluation Summary

| GEO Accession | Platform | Total Samples | Healthy Controls ($n$) | Cirrhosis Samples ($n$) | Confounding Groups / Exclusions | Meets $\ge 15$ Per Group Rule? | Verdict |
| :--- | :--- | :---: | :---: | :---: | :--- | :---: | :--- |
| **GSE14323** | GPL570 | 124 | **19** | **58** | 47 liver tissues with HCC (can be filtered) | **Yes** | **Selected** — Best sample balance & clear non-tumor subset. |
| **GSE139602** | GPL13667 | 39 | 6 | 20 | 5 eCLD, 8 ACLF; progressive disease staging | **No** | *Underpowered* — Healthy control group ($n=6$) is too small. |
| **GSE25097** | GPL10687 | 557 | 6 | 40 | 268 tumor, 243 non-tumor | **No** | *Underpowered* — Healthy control group ($n=6$) is too small. |
| **GSE6764** | GPL570 | 75 | 10 | 13 | 52 dysplasia / HCC stages | **No** | *Underpowered* — Neither group reaches $n=15$. |

## **GSE14323 Selected** - Best sample balance & clear non-tumor subset.

## Data Preprocessing and Quality Control

In [8]:
gse = GEOparse.get_GEO(geo="GSE14323", destdir="./geodata", annotate_gpl=True)

05-Nov-2026 09:45:11 DEBUG utils - Directory ./geodata already exists. Skipping.
DEBUG:GEOparse:Directory ./geodata already exists. Skipping.
05-Nov-2026 09:45:11 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
05-Nov-2026 09:45:11 INFO GEOparse - Parsing ./geodata/GSE14323_family.soft.gz: 
INFO:GEOparse:Parsing ./geodata/GSE14323_family.soft.gz: 
05-Nov-2026 09:45:11 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
05-Nov-2026 09:45:11 DEBUG GEOparse - SERIES: GSE14323
DEBUG:GEOparse:SERIES: GSE14323
05-Nov-2026 09:45:11 DEBUG GEOparse - PLATFORM: GPL96
DEBUG:GEOparse:PLATFORM: GPL96
05-Nov-2026 09:45:11 DEBUG GEOparse - PLATFORM: GPL571
DEBUG:GEOparse:PLATFORM: GPL571
05-Nov-2026 09:45:12 DEBUG GEOparse - SAMPLE: GSM358103
DEBUG:GEOparse:SAMPLE: GSM358103
05-Nov-2026 09:45:12 DEBUG GEOparse - SAMPLE: GSM358104
DEBUG:GEOparse:SAMPLE: GSM358104
05-Nov-2026 09:45:12 DEBUG GEOparse - SAMPLE: GSM358105
DE

In [9]:
## initial samples
initial_samples = len(gse.gsms)
print(f"Total samples: {initial_samples}")

Total samples: 124


In [18]:
expression_frames =[]
meta_list = []

for gsm_name, gsm in gse.gsms.items():
    source = gsm.metadata.get("source_name_ch1",[ ''])[0]
    # filtering out the 47 GCC tissue samples, keeping only healthy and cirrhosis samples
    if 'hcc' in source.lower():
        continue
    group = 'healthy' if any(term in source.lower() for term in ['normal', 'healthy', 'control']) else 'cirrhosis'
    meta_list.append({'sample':gsm_name, 'source':source, 'group':group})
    df = gsm.table.set_index('ID_REF')['VALUE']
    expression_frames.append(df)

meta_df = pd.DataFrame(meta_list).set_index('sample')
expression_matrix = pd.concat(expression_frames, axis=1)
initial_probes = expression_matrix.shape[0]
print(f"Total probes before filtering: {initial_probes}")
print(f"Retained samples after filtering: {len(meta_df)} (Healthy:{sum(meta_df['group']=='healthy')}, Cirrhosis:{sum(meta_df['group']=='cirrhosis')})")




Total probes before filtering: 22277
Retained samples after filtering: 60 (Healthy:19, Cirrhosis:41)


In [19]:
# Normalization and Transformation check
min_val, max_val = expression_matrix.min().min(), expression_matrix.max().max()
print(f"Expression value range: [{min_val:.2f}, {max_val:.2f}]")
if max_val > 50:
    # values are not log-transformed therefore log-transform
    expression_matrix = np.log2(expression_matrix + 1)
    print("Applying log2 transformation to expression values.")
else:
    print("Expression values are already log-transformed.")

Expression value range: [3.48, 14.36]
Expression values are already log-transformed.


In [20]:
# handling missing values and low-expression filtering

expression_matrix = expression_matrix.dropna()
variance_threshold = expression_matrix.var(axis=1).quantile(0.10) # drop bottom 10% lowest variance genes
expression_matrix = expression_matrix[expression_matrix.var(axis=1) > variance_threshold]
print(f" Remaining genes: {expression_matrix.shape[0]}")

 Remaining genes: 20049


In [21]:
print(expression_matrix.shape)

(20049, 60)


reduced feature space from 22,277 down to 20,049 genes